# Dancing with the SARs – Douala : finalisation des données (D13 → D17)

Ce notebook part des PNG et du `stats_s1.csv` déjà produits (Sentinel-1, VV, produits par le notebook D5) et fabrique
ce que le front consomme :

- `data/layers/` : PNG radar + PNG eau (eau en bleu semi-transparent, le reste transparent)
- `data/water_series.json` : `{date, surface_km2, backscatter_db}` (contrat T0.4)
- `data/manifest.json` : une entrée par date avec `date, saison, backscatter_png, water_png, coordinates`
- `data/composites.json` : composites saison sèche / pluies (pour le bouton F12)
- `douala_data.zip` : tout le dossier `data/`, à décompresser dans `frontend/public/`

**Avant de lancer** : dans `SAR_Douala/raw/` sur Drive, il doit y avoir les 37 images, `stats_s1.csv`
et `douala_aoi.geojson` (l'emprise de la tâche D1). Ne mettez aucun jeton ni mot de passe dans ce notebook.

In [ ]:
# 1. Configuration et accès au Drive
import json, shutil, re
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/SAR_Douala')
    IN_COLAB = True
except ImportError:          # exécution locale
    BASE = Path('SAR_Douala')
    IN_COLAB = False

RAW = BASE / 'raw'
OUT = BASE / 'output' / 'data'
LAYERS = OUT / 'layers'
LAYERS.mkdir(parents=True, exist_ok=True)

# Chemin public tel que le front le verra (Vite sert public/ à la racine)
PUBLIC_PREFIX = '/data/layers/'

# Couleur de l'eau (D14) : bleu semi-transparent RGBA
WATER_RGBA = (34, 211, 238, 170)   # cyan du contrat v1.0 (même couleur que le notebook D5)
MAX_MB = 2.0
print('Dossier de travail :', BASE)
print('Fichiers dans raw/ :', len(list(RAW.glob('*'))))

## 2. Emprise de la zone (les 4 coins pour MapLibre)
Les PNG n'ont pas de coordonnées. Ils ont été exportés (notebook D5) sur la BBOX du contrat : 9.55, 3.90, 9.90, 4.20, en 1400 × 1200 px, EPSG:4326. On la renseigne donc à la main dans `BBOX_MANUELLE` (le GeoJSON n'est plus nécessaire).

In [ ]:
# 2. Lecture de l'emprise
BBOX_MANUELLE = (9.55, 3.90, 9.90, 4.20)   # (ouest, sud, est, nord) = BBOX du notebook D5 (contrat v1.0) ; None -> lit douala_aoi.geojson

def all_coords(obj):
    if isinstance(obj, (list, tuple)):
        if len(obj) >= 2 and all(isinstance(v, (int, float)) for v in obj[:2]):
            yield obj[:2]
        else:
            for o in obj:
                yield from all_coords(o)
    elif isinstance(obj, dict):
        for v in obj.values():
            yield from all_coords(v)

if BBOX_MANUELLE:
    W, S, E, N = BBOX_MANUELLE
else:
    gj = json.loads((RAW / 'douala_aoi.geojson').read_text())
    pts = np.array(list(all_coords(gj.get('features', gj))))
    W, S, E, N = (float(v) for v in (pts[:, 0].min(), pts[:, 1].min(), pts[:, 0].max(), pts[:, 1].max()))

# Ordre exigé par MapLibre : haut-gauche, haut-droite, bas-droite, bas-gauche (lon, lat)
COORDS = [[W, N], [E, N], [E, S], [W, S]]
print('Ouest/Sud/Est/Nord :', W, S, E, N)
print('Coins :', COORDS)
assert 8 < W < 11 and 3 < S < 5, "L'emprise ne ressemble pas à Douala : vérifiez le GeoJSON"


## 3. Lecture du CSV et inventaire des images

In [ ]:
# 3. CSV + vérification des fichiers
df = pd.read_csv(RAW / 'stats_s1.csv')
df.columns = [c.strip() for c in df.columns]
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
df = df.sort_values('date').reset_index(drop=True)

manquants = []
for d in df['date']:
    for kind in ('backscatter_vv', 'water_mask'):
        if not (RAW / f'{d}_{kind}.png').exists():
            manquants.append(f'{d}_{kind}.png')
print(len(df), 'dates dans le CSV ;', 'fichiers manquants :', manquants or 'aucun')
assert not manquants

# Toutes les images doivent avoir la même taille (sinon les coins ne conviennent pas à toutes)
tailles = {}
for d in df['date']:
    for kind in ('backscatter_vv', 'water_mask'):
        im = Image.open(RAW / f'{d}_{kind}.png')
        tailles.setdefault((im.size, im.mode, kind), []).append(d)
for k, v in tailles.items():
    print(k, '->', len(v), 'images')
assert len({k[0] for k in tailles}) == 1, 'Les images n ont pas toutes la même dimension !'
df.head()

## 4. D13 / D14 – PNG radar et PNG eau prêts pour la carte

In [ ]:
# 4. Conversion des couches
def save_png(img, path):
    img.save(path, optimize=True)
    return path.stat().st_size / 1e6

def to_backscatter(path_in, path_out):
    im = Image.open(path_in)
    if im.mode in ('RGBA', 'LA'):
        im = im.convert('RGBA')
    else:
        im = im.convert('L')
    return save_png(im, path_out)

def to_water(path_in, path_out):
    im = Image.open(path_in)
    arr = np.array(im.convert('RGBA'))
    alpha = arr[..., 3]
    if (alpha == 0).any():                 # le masque a déjà une transparence
        water = alpha > 0
    else:                                  # sinon : fond sombre = pas d'eau
        water = arr[..., :3].max(axis=2) > 40
    out = np.zeros(arr.shape, dtype=np.uint8)
    out[water] = WATER_RGBA
    save_png(Image.fromarray(out), path_out)
    return water.sum()

lignes = []
for d in df['date']:
    mb = to_backscatter(RAW / f'{d}_backscatter_vv.png', LAYERS / f'{d}_backscatter_vv.png')
    npx = to_water(RAW / f'{d}_water_mask.png', LAYERS / f'{d}_water_mask.png')
    lignes.append((d, npx, mb))
chk = pd.DataFrame(lignes, columns=['date', 'pixels_eau', 'poids_radar_Mo'])
print('Poids max d une image radar :', round(chk['poids_radar_Mo'].max(), 2), 'Mo (objectif < 2)')
chk.head()

## 5. Contrôle de cohérence masque ↔ CSV
`surface_km2 / pixels_eau` doit être à peu près constant (même résolution, même zone). Une date qui s'écarte
beaucoup signale un masque différent de celui qui a servi au calcul du CSV.

In [ ]:
# 5. Cohérence
chk = chk.merge(df[['date', 'water_area_km2']], on='date')
chk['km2_par_pixel'] = chk['water_area_km2'] / chk['pixels_eau'].replace(0, np.nan)
med = chk['km2_par_pixel'].median()
chk['ecart_%'] = ((chk['km2_par_pixel'] / med - 1) * 100).round(1)
print('km² par pixel (médiane) :', round(med, 6))
suspects = chk[chk['ecart_%'].abs() > 10]
print('Dates suspectes (écart > 10 %) :', list(suspects['date']) or 'aucune')
chk[['date', 'pixels_eau', 'water_area_km2', 'ecart_%']]

## 6. D15 / D16 – `water_series.json`, `manifest.json`, composites

In [ ]:
# 6. JSON du contrat T0.4
series = [
    {'date': r.date,
     'surface_km2': round(float(r.water_area_km2), 2),
     'backscatter_db': round(float(r.mean_vv_db), 2)}
    for r in df.itertuples()
]
(OUT / 'water_series.json').write_text(json.dumps(series, indent=2, ensure_ascii=False))

manifest = [
    {'date': r.date,
     'saison': r.saison,                       # seche | pluies | controle
     'satellite': r.satellite,
     'backscatter_png': f'{PUBLIC_PREFIX}{r.date}_backscatter_vv.png',
     'water_png': f'{PUBLIC_PREFIX}{r.date}_water_mask.png',
     'coordinates': COORDS}
    for r in df.itertuples()
]
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

# Composites saison sèche / pluies (hors contrat : fichier séparé, pour le bouton F12)
comp = {}
for saison in ('seche', 'pluies'):
    b, w = RAW / f'composite-{saison}_backscatter_vv.png', RAW / f'composite-{saison}_water_mask.png'
    if b.exists() and w.exists():
        to_backscatter(b, LAYERS / f'composite-{saison}_backscatter_vv.png')
        to_water(w, LAYERS / f'composite-{saison}_water_mask.png')
        comp[saison] = {'backscatter_png': f'{PUBLIC_PREFIX}composite-{saison}_backscatter_vv.png',
                        'water_png': f'{PUBLIC_PREFIX}composite-{saison}_water_mask.png',
                        'coordinates': COORDS}
(OUT / 'composites.json').write_text(json.dumps(comp, indent=2, ensure_ascii=False))
# NB : composite-pluies_new_water_mask.png est volontairement ignoré (image quasi vide)

print(len(series), 'points dans la série ;', len(manifest), 'entrées dans le manifeste ;', list(comp), 'composites')
print(json.dumps(manifest[0], indent=2))

## 7. Vérification visuelle (D12) et courbe

In [ ]:
# 7. Coup d'œil
import matplotlib.pyplot as plt

couleurs = {'seche': '#d98c1f', 'pluies': '#1f6fd9', 'controle': '#888888'}
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(pd.to_datetime(df['date']), df['water_area_km2'], color='#999', zorder=1)
for s, c in couleurs.items():
    sub = df[df['saison'] == s]
    ax.scatter(pd.to_datetime(sub['date']), sub['water_area_km2'], color=c, label=s, zorder=2)
ax.set_ylabel('Surface en eau (km²)'); ax.legend(); ax.grid(alpha=.3)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, d in zip(axes, [df['date'].iloc[0], df['date'].iloc[-1]]):
    ax.imshow(Image.open(LAYERS / f'{d}_backscatter_vv.png'), cmap='gray')
    ax.imshow(Image.open(LAYERS / f'{d}_water_mask.png'))
    ax.set_title(d); ax.axis('off')
plt.show()

print(df.groupby('saison')['water_area_km2'].agg(['count', 'mean', 'min', 'max']).round(1))

## 8. D17 – Archive à déposer dans le dépôt

In [ ]:
# 8. Zip prêt pour GitHub : à décompresser dans frontend/public/ (crée public/data/...)
poids = sum(f.stat().st_size for f in OUT.rglob('*') if f.is_file()) / 1e6
print('Poids total de data/ :', round(poids, 1), 'Mo')
zip_path = shutil.make_archive(str(BASE / 'output' / 'douala_data'), 'zip',
                               root_dir=OUT.parent, base_dir='data')
print('Archive :', zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)